In [ ]:
# ====================================================================
# CLASSIFICATION MODEL TRAINING
# ====================================================================
# Trains a CNN to classify left and right kinetic model types
# from thermogravimetric (TG) curves

import os

# List available input data files
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# ====================================================================
# LOAD TRAINING DATA FROM DISK
# ====================================================================
# Retrieve previously generated synthetic dataset

import pickle

with open("/kaggle/input/curves-data-generation-a/data_classification.pkl", "rb") as f:
#with open("/kaggle/input/curves-v1/data_classification.pkl", "rb") as f:
    data = pickle.load(f)

# Unpack training, validation, and test sets
X_train = data["X_train"]
y_train = data["y_train"]
y_left_train = data["y_left_train"]
y_right_train = data["y_right_train"]
X_val = data["X_val"]
y_val = data["y_val"]
y_left_val = data["y_left_val"]
y_right_val = data["y_right_val"]
X_test = data["X_test"]
y_test = data["y_test"]
y_left_test = data["y_left_test"]
y_right_test = data["y_right_test"]
mix_min = data['mins'][0, 2]
mix_range = data['ranges'][0, 2]

# Unpack configuration
p = data["p"]
q = data["q"]
r = data["r"]
params = data["params"]
BATCH_SIZE = data["BATCH_SIZE"]
STEPS_PER_EPOCH = data["STEPS_PER_EPOCH"]
VALIDATION_STEPS = data["VALIDATION_STEPS"]
TEST_STEPS = data["TEST_STEPS"]
SEGMENTS = data["SEGMENTS"]
EPOCHS = data["EPOCHS"]

r = 7

In [ ]:
# ====================================================================
# PREPROCESS TG SPECTRA
# ====================================================================
# Build derivative channels after loading clean TG spectra.

import numpy as np

DERIVATIVE_MODE = "all"  # "first", "first_second", or "all"
NOISE_LEVEL = 0.00  # 0.00, 0.01, 0.02, or 0.05; applied to test only
NOISE_SCALING = "per_curve_range"  # "per_curve_range", "global_range", or "absolute"
NOISE_SEED = 42

if DERIVATIVE_MODE not in {"first", "first_second", "all"}:
    raise ValueError("DERIVATIVE_MODE must be 'first', 'first_second', or 'all'")
if NOISE_LEVEL not in {0.00, 0.01, 0.02, 0.05}:
    raise ValueError("NOISE_LEVEL must be 0.00, 0.01, 0.02, or 0.05")
if NOISE_SCALING not in {"per_curve_range", "global_range", "absolute"}:
    raise ValueError("Unsupported NOISE_SCALING")

def add_noise(curves, noise_level=0.0, scaling="per_curve_range", seed=None):
    """Add zero-mean Gaussian noise to TG spectra."""
    curves = np.asarray(curves, dtype=np.float32)
    if noise_level == 0.0:
        return curves.copy()
    rng = np.random.default_rng(seed)
    if scaling == "per_curve_range":
        scale = np.ptp(curves, axis=1, keepdims=True)
    elif scaling == "global_range":
        scale = np.ptp(curves)
    elif scaling == "absolute":
        scale = 1.0
    else:
        raise ValueError("Unsupported noise scaling")
    return curves + rng.normal(0.0, noise_level * scale, size=curves.shape).astype(np.float32)

def build_features(curves, temperature, temp_step, derivative_mode="all"):
    """Convert TG spectra into the selected derivative feature channels."""
    curves = np.asarray(curves, dtype=np.float32)
    first = np.diff(curves, axis=1, prepend=curves[:, :1]) / temp_step
    if derivative_mode == "first":
        return first[..., None]
    if derivative_mode == "first_second":
        second = np.diff(first, axis=1, prepend=first[:, :1]) / temp_step
        return np.stack([first, second], axis=-1)
    if derivative_mode == "all":
        second = np.diff(first, axis=1, prepend=first[:, :1]) / temp_step
        scaled_first = first * np.asarray(temperature, dtype=np.float32)[None, :] ** 2
        return np.stack([first, second, scaled_first], axis=-1)
    raise ValueError("Unsupported derivative mode")

temperature = data["temperature"]
temp_step = data["temp_step"]
X_train = build_features(X_train, temperature, temp_step, DERIVATIVE_MODE)
X_val = build_features(X_val, temperature, temp_step, DERIVATIVE_MODE)
X_test = build_features(
    add_noise(X_test, NOISE_LEVEL, NOISE_SCALING, NOISE_SEED),
    temperature,
    temp_step,
    DERIVATIVE_MODE,
)
INPUT_CHANNELS = X_train.shape[-1]
print(f"Features: {DERIVATIVE_MODE}, test noise: {NOISE_LEVEL:.0%} ({NOISE_SCALING})")

In [ ]:
# ====================================================================
# REQUIRED IMPORTS FOR MODEL TRAINING
# ====================================================================

import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, BatchNormalization, Flatten, Dense, Reshape
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LearningRateScheduler, EarlyStopping
import matplotlib.pyplot as plt
import time

In [ ]:
# ====================================================================
# CONFIGURATION: Training Parameters
# ====================================================================
# Hyperparameters for model training

BATCH_SIZE = 256  # Batch size (from optimization)
EPOCHS = 500 #100  # Number of training epochs

In [ ]:
# ====================================================================
# CREATE TRAINING PIPELINE AND BUILD MODEL
# ====================================================================
# Step 1: Create tf.data.Dataset for training and validation
# Step 2: Define CNN architecture
# Step 3: Configure training

# Create training dataset with shuffling, batching, and prefetching
train_dataset = (
    tf.data.Dataset
    .from_tensor_slices((X_train, {'left_output': y_left_train, 'right_output': y_right_train}))
    .shuffle(buffer_size=BATCH_SIZE*5)
    .batch(BATCH_SIZE)
    .prefetch(1)
)

# Create validation dataset
validation_dataset = (
    tf.data.Dataset
    .from_tensor_slices((X_val, {'left_output': y_left_val, 'right_output': y_right_val}))
    .shuffle(buffer_size=BATCH_SIZE*2)
    .batch(BATCH_SIZE)
    .prefetch(1)
)

In [ ]:
# ====================================================================
# CLEANUP: Free memory from raw arrays after creating tf.data pipelines
# ====================================================================
# The tf.data.Dataset objects hold references to the data, so raw arrays can be deleted

import gc

print("Cleaning up training data from memory...")

# Delete raw training and validation arrays (not needed anymore - they're in tf.data pipelines)
del X_train, y_train, y_left_train, y_right_train
del X_val, y_val, y_left_val, y_right_val
del data  # Original pickle data dict
del params  # Not needed until inference (will reload from pickle if needed)

# Garbage collection to ensure memory is freed
gc.collect()
if tf.config.list_physical_devices('GPU'):
    tf.keras.backend.clear_session()

print("✓ Freed training/validation arrays and metadata dicts")
print("✓ X_test, y_test kept for inference")

In [ ]:
# ====================================================================
# BUILD CNN MODEL ARCHITECTURE
# ====================================================================
# Input: selected derivative channels from the TG spectrum
# Output: Two classification heads (left and right model types)

first = Input(shape=(p, INPUT_CHANNELS))
x = Reshape((p, INPUT_CHANNELS))(first)
x = Conv1D(32, 9, strides=1, activation="relu")(x)
x = BatchNormalization()(x)
x = Flatten()(x)
x = Dense(512, activation="relu")(x)
x = BatchNormalization()(x)

# Two output heads: classify left and right model types
y_left = Dense(r, activation="softmax", name="left_output")(x)
y_right = Dense(r, activation="softmax", name="right_output")(x)

model = Model(inputs=first, outputs=[y_left, y_right])

# ====================================================================
# CONFIGURE TRAINING: CALLBACKS AND COMPILATION
# ====================================================================

# Learning rate scheduler: decay rate after epoch 10 and 50
def scheduler(epoch, lr):
    if epoch < 50:
        return lr
    elif epoch < 130:
        return lr * 0.97
    elif epoch < 210:
        return lr * 0.98
    elif epoch < 290:
        return lr * 0.99
    else:
        return lr

lr_callback = LearningRateScheduler(scheduler)
es_callback = EarlyStopping(monitor='val_loss', patience=100, restore_best_weights=True)

# Compile model with categorical cross-entropy loss for both outputs
model.compile(
    optimizer=Adam(learning_rate=1.833602742464206e-05),
    loss={
        'left_output': 'sparse_categorical_crossentropy',
        'right_output': 'sparse_categorical_crossentropy'
    },
    metrics={'left_output': 'accuracy', 'right_output': 'accuracy'}
)

model.summary()

In [ ]:
# ====================================================================
# TRAIN THE MODEL
# ====================================================================

print("\nStarting model training...")
start_time = time.time()

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=validation_dataset,
    verbose=1,
    callbacks=[lr_callback, es_callback]
)

end_time = time.time()
print("✓ Training complete.")
print(f"Training time: {end_time - start_time:.2f} seconds")

In [ ]:
# ====================================================================
# PLOT TRAINING HISTORY - Model Accuracy
# ====================================================================
# Visualize training and validation accuracy over epochs

plt.figure(figsize=(12, 5))
plt.plot(history.history['left_output_accuracy'], label='Left Training Accuracy', linewidth=2, linestyle="--")
plt.plot(history.history['right_output_accuracy'], label='Right Training Accuracy', linewidth=2, linestyle="--")
plt.plot(history.history['val_left_output_accuracy'], label='Left Validation Accuracy', linewidth=2)
plt.plot(history.history['val_right_output_accuracy'], label='Right Validation Accuracy', linewidth=2)
plt.title('Model Classification Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim((0.9, 1))
plt.tight_layout()
plt.show()

print("✓ Accuracy plot generated")

In [ ]:
# ====================================================================
# TEST SET INFERENCE
# ====================================================================
# Make predictions on test data

print("Generating predictions on test set...")
y_pred = model.predict(X_test)
print("✓ Predictions complete")

In [ ]:
# ====================================================================
# EVALUATE TEST SET PERFORMANCE
# ====================================================================
# Compute confusion matrices and accuracy for left/right classification

from sklearn.metrics import confusion_matrix, accuracy_score
import pandas as pd

# Model names for reference
MODEL_NAMES = ['D2', 'D3', 'D4', 'R2', 'R3', 'Fn', 'JMA']

# ---- LEFT COMPONENT ----
# Get predicted classes (argmax of softmax outputs)
left_pred_classes = np.argmax(y_pred[0], axis=1)
left_confusion = confusion_matrix(y_left_test, left_pred_classes)
left_confusion_norm = confusion_matrix(y_left_test, left_pred_classes, normalize="true")

# Create DataFrames for display
df_left = pd.DataFrame(left_confusion, columns=MODEL_NAMES, index=MODEL_NAMES)
df_left_norm = pd.DataFrame(left_confusion_norm, columns=MODEL_NAMES, index=MODEL_NAMES)

left_accuracy = accuracy_score(y_left_test, left_pred_classes)

# ---- RIGHT COMPONENT ----
right_pred_classes = np.argmax(y_pred[1], axis=1)
right_confusion = confusion_matrix(y_right_test, right_pred_classes)
right_confusion_norm = confusion_matrix(y_right_test, right_pred_classes, normalize="true")

df_right = pd.DataFrame(right_confusion, columns=MODEL_NAMES, index=MODEL_NAMES)
df_right_norm = pd.DataFrame(right_confusion_norm, columns=MODEL_NAMES, index=MODEL_NAMES)

right_accuracy = accuracy_score(y_right_test, right_pred_classes)

# ---- PRINT RESULTS ----
print("\n" + "="*80)
print("LEFT COMPONENT - CONFUSION MATRIX (Raw Counts)")
print("="*80)
print(df_left)

print("\n" + "="*80)
print("LEFT COMPONENT - CONFUSION MATRIX (Normalized)")
print("="*80)
print(df_left_norm.round(2))
print(f"\nLeft Accuracy: {left_accuracy:.4f}")

print("\n" + "="*80)
print("RIGHT COMPONENT - CONFUSION MATRIX (Raw Counts)")
print("="*80)
print(df_right)

print("\n" + "="*80)
print("RIGHT COMPONENT - CONFUSION MATRIX (Normalized)")
print("="*80)
print(df_right_norm.round(2))
print(f"\nRight Accuracy: {right_accuracy:.4f}")
print("="*80)

In [ ]:
# ====================================================================
# VISUALIZE CONFUSION MATRICES - Heatmaps
# ====================================================================
# Display normalized confusion matrices as heatmaps for visual analysis

import seaborn as sns

print("\nGenerating confusion matrix heatmaps...\n")

# Left component heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df_left_norm, annot=True, fmt='.2f', cmap="Blues", vmax=1, cbar_kws={'label': 'Accuracy'})
plt.title('Left Component - Classification Accuracy Heatmap')
plt.xlabel('Predicted Model')
plt.ylabel('True Model')
plt.tight_layout()
plt.show()

# Right component heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df_right_norm, annot=True, fmt='.2f', cmap="Blues", vmax=1, cbar_kws={'label': 'Accuracy'})
plt.title('Right Component - Classification Accuracy Heatmap')
plt.xlabel('Predicted Model')
plt.ylabel('True Model')
plt.tight_layout()
plt.show()

print("✓ Heatmaps generated")

In [ ]:
from scipy.ndimage import gaussian_filter1d

mix_ratio = mix_min + y_test[:, 2] * mix_range

MODEL_NAMES = ['D2','D3','D4','R2','R3','Fn','JMA']
bin_edges = np.linspace(mix_min, mix_min + mix_range, 25)

def smoothed_accuracy(correct, mix_values, bin_edges, sigma=1.2):
    centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    if mix_values.size == 0:
        return centers, np.full_like(centers, np.nan, dtype=float)
    digitized = np.digitize(mix_values, bin_edges) - 1
    stats = np.full_like(centers, np.nan, dtype=float)
    for idx in range(centers.size):
        mask = digitized == idx
        if mask.any():
            stats[idx] = correct[mask].mean()
    valid = ~np.isnan(stats)
    filled = stats.copy()
    if valid.sum() >= 2:
        filled[~valid] = np.interp(centers[~valid], centers[valid], stats[valid])
        return centers, gaussian_filter1d(filled, sigma=sigma)
    return centers, stats

fig, axes = plt.subplots(len(MODEL_NAMES), 2, figsize=(12, len(MODEL_NAMES) * 2.3), sharex=True, sharey=True)
for idx, model in enumerate(MODEL_NAMES):
    left_mask = y_left_test == idx
    right_mask = y_right_test == idx
    correct_left = (left_pred_classes[left_mask] == y_left_test[left_mask]).astype(float)
    correct_right = (right_pred_classes[right_mask] == y_right_test[right_mask]).astype(float)

    centers_l, curve_l = smoothed_accuracy(correct_left, mix_ratio[left_mask], bin_edges)
    centers_r, curve_r = smoothed_accuracy(correct_right, mix_ratio[right_mask], bin_edges)

    ax_left = axes[idx, 0]
    ax_left.plot(centers_l, curve_l, color='#1f77b4', lw=2)
    ax_left.scatter(mix_ratio[left_mask], correct_left, s=10, alpha=0.2, color='#1f77b4')
    ax_left.set_title(f'Left true {model}')
    ax_left.set_ylabel('Accuracy')
    ax_left.grid(alpha=0.3)
    ax_left.set_ylim(0, 1)

    ax_right = axes[idx, 1]
    ax_right.plot(centers_r, curve_r, color='#ff7f0e', lw=2)
    ax_right.scatter(mix_ratio[right_mask], correct_right, s=10, alpha=0.2, color='#ff7f0e')
    ax_right.set_title(f'Right true {model}')
    ax_right.grid(alpha=0.3)
    ax_right.set_ylim(0, 1)

for ax in axes[-1, :]:
    ax.set_xlabel('Mix ratio')

fig.suptitle('Accuracy vs mix ratio for each true kinetic class', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# SAVE TRAINED CLASSIFICATION MODEL
# ====================================================================
# Persist the trained model for use in downstream inference

import os

CLASSIFICATION_MODEL_PATH = '/kaggle/working/cnn_classification_model.h5'

model.save(CLASSIFICATION_MODEL_PATH)

print(f"✓ Classification model saved to: {CLASSIFICATION_MODEL_PATH}")
print(f"  File size: {os.path.getsize(CLASSIFICATION_MODEL_PATH) / 1024 / 1024:.2f} MB")